# **CNN과 비전 객체탐지**


---



**(코랩에서)한글 폰트 지정하는 방법**

In [ ]:
# 코랩에서 실행
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import shutil
import warnings
import logging

# ── 1. 나눔 폰트 설치 ──────────────────────────────────────────
!apt-get install -y fonts-nanum > /dev/null 2>&1

# ── 2. matplotlib 캐시 디렉터리 삭제 ──────────────────────────
shutil.rmtree(mpl.get_cachedir(), ignore_errors=True)

# ── 3. 폰트 등록 및 유니코드 마이너스 경고 방지 ──────────────────
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (6, 4)

# 수학 기호(마이너스 등)를 일반 텍스트로 처리하도록 설정하여 폰트 에러 차단
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.default'] = 'regular'

# mathtext 및 font_manager 관련 폰트 결함 경고 완전히 무시 설정
logging.getLogger('matplotlib.mathtext').setLevel(logging.ERROR)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib.font_manager')



---



# **1. 이미지와 합성곱의 기초**

- 목표
    - ① 이미지가 숫자 배열(텐서)임을 확인
    - ② 합성곱을 손으로 계산하고 PyTorch와 비교
    - ③ 출력 크기 공식
    - ④ 사람이 설계한 필터(Sobel·Canny·템플릿매칭)의 장점과 한계 |
- 필요 패키지
    - `torch`, `numpy`, `matplotlib`, `opencv-python`, `scikit-image` (Colab에 기본 설치)


## 0. 환경 설정

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2

def set_korean_font():
    """설치된 한글 폰트를 찾아 matplotlib 기본 폰트로 지정합니다."""
    candidates = ["Malgun Gothic", "NanumGothic", "AppleGothic", "Noto Sans CJK KR"]
    installed = {f.name for f in fm.fontManager.ttflist}
    for name in candidates:
        if name in installed:
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False   # 마이너스 기호 깨짐 방지

set_korean_font()
torch.manual_seed(42)
np.random.seed(42)
print("PyTorch:", torch.__version__, "| OpenCV:", cv2.__version__)

## 1-1. 이미지 패턴 인식의 발전

### **1. 이미지는 숫자다** — 텐서로 바라보기

| 표현 | 모양(shape) | 사용처 |
|---|---|---|
| 흑백 이미지 | (H, W) | OpenCV, NumPy |
| 컬러 이미지 | (H, W, C) — **HWC** | OpenCV(BGR 순서!), PIL, matplotlib(RGB) |
| PyTorch 입력 | (N, C, H, W) — **NCHW** | `nn.Conv2d` |

`scikit-image`에 내장된 고양이 사진(chelsea)을 사용합니다. 인터넷 다운로드가 필요 없습니다.

In [ ]:
from skimage import data

cat = data.chelsea()            # RGB 컬러 고양이 사진, NumPy 배열
print("모양(H, W, C):", cat.shape, "| 자료형:", cat.dtype, "| 값 범위:", cat.min(), "~", cat.max())

# 가운데 8×8 픽셀만 숫자로 출력 → '이미지 = 숫자 배열'을 눈으로 확인
print("빨강(R) 채널 일부 8×8:\n", cat[100:108, 200:208, 0])

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
axes[0].imshow(cat)
axes[0].set_title("원본 RGB")
for i, (ax, name, cmap) in enumerate(zip(axes[1:], ["R", "G", "B"], ["Reds", "Greens", "Blues"])):
    ax.imshow(cat[:, :, i], cmap=cmap)
    ax.set_title(f"{name} 채널")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# HWC(uint8, 0~255)  →  NCHW(float32, 0~1) : PyTorch 모델에 넣기 위한 표준 변환
x = torch.from_numpy(cat)                # (H, W, C)
x = x.permute(2, 0, 1).float() / 255.0   # (C, H, W):채널우선구조 + 0.0~1.0으로 정규화
x = x.unsqueeze(0)                       # (N=1, C, H, W)
print("PyTorch 텐서 모양:", tuple(x.shape), "| 값 범위:", round(x.min().item(), 2), "~", round(x.max().item(), 2))



---



### **2. 이미지 패턴 인식의 세 가지 접근 방법**

#### **🟠 구조적 접근법** — Canny 에지 + 규칙
사람이 만든 규칙: ①수평으로 나란한 ②적당히 떨어진 ③크기가 비슷한 ④위쪽의 원 2개 = 눈, ⑤그 아래 = 코

In [ ]:
# ============================================================
# [예제 1] 구조적 접근법 — Canny 에지(1986) + 사람이 만든 규칙
#   목표: "고양이 얼굴 = 비슷한 크기의 원(눈) 2개가 수평으로 나란히 있고,
#          그 아래 가운데에 코가 있다"는 '문법(규칙)'으로 얼굴 구조를 찾는다.
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color, feature, transform

cat = data.chelsea()                       # (300, 451, 3) uint8
gray = color.rgb2gray(cat)                 # 0.0~1.0 흑백 (에지는 '밝기 변화'만 본다)
RADII = np.arange(22, 42, 2)               # 찾을 눈 반지름 후보(픽셀)

def is_eye_pair(c1, c2, img_h):
    """사람이 직접 정한 규칙 4개 — 모두 만족해야 '두 눈'으로 인정"""
    (x1, y1, r1, _), (x2, y2, r2, _) = c1, c2
    rule_horizontal = abs(y1 - y2) < 50                  # 규칙1: 거의 같은 높이
    rule_distance   = 100 < abs(x1 - x2) < 250           # 규칙2: 적당히 떨어져 있음
    rule_same_size  = max(r1, r2) / min(r1, r2) < 1.4    # 규칙3: 크기가 비슷
    rule_upper_half = max(y1, y2) < img_h * 0.6          # 규칙4: 얼굴 위쪽에 위치
    return rule_horizontal and rule_distance and rule_same_size and rule_upper_half

def detect_face_by_rules(gray):
    """① Canny 에지 → ② Hough 원 후보 → ③ 규칙으로 눈 쌍 선택 → ④ 규칙으로 코 추론"""
    # ① 기본 요소(Primitive) 추출: sigma가 클수록 잔털·수염 같은 잡음 에지가 줄어든다
    edges = feature.canny(gray, sigma=3.0, low_threshold=0.05, high_threshold=0.15)
    # ② 에지에서 '원' 모양 후보 찾기 (Hough 원 변환: 에지 점들이 투표해 원 중심을 찾음)
    h = transform.hough_circle(edges, RADII)
    acc, cx, cy, rr = transform.hough_circle_peaks(
        h, RADII, total_num_peaks=12, min_xdistance=30, min_ydistance=30)
    circles = list(zip(cx, cy, rr, acc))
    # ③ 규칙을 통과한 쌍 중 '원다움(허프 누적값)' 합이 가장 큰 쌍 = 두 눈
    best, best_score = None, -1
    for i in range(len(circles)):
        for j in range(i + 1, len(circles)):
            if is_eye_pair(circles[i], circles[j], gray.shape[0]):
                s = circles[i][3] + circles[j][3]
                if s > best_score:
                    best, best_score = (circles[i], circles[j]), s
    # ④ 규칙5: 코 = 두 눈 중점에서 '눈 사이 거리 × 0.9' 만큼 아래
    nose = None
    if best:
        (x1, y1, *_), (x2, y2, *_) = best
        nose = ((x1 + x2) / 2, (y1 + y2) / 2 + 0.9 * abs(x2 - x1))
    return edges, circles, best, nose

def draw_result(ax, img, circles, best, nose, title):
    ax.imshow(img, cmap="gray")
    for (x, y, r, _) in circles:            # 흰 점선: 규칙 적용 전 '원 후보' 전부
        ax.add_patch(plt.Circle((x, y), r, fill=False, ec="white", ls=":", lw=1))
    if best:
        for (x, y, r, _) in best:           # 초록: 규칙을 통과한 '두 눈'
            ax.add_patch(plt.Circle((x, y), r, fill=False, ec="lime", lw=3))
        ax.plot(*nose, "r^", ms=14)         # 빨강: 규칙으로 추론한 코
    ax.set_title(title); ax.axis("off")

# ---------- 원본 사진에 적용 ----------
edges, circles, best, nose = detect_face_by_rules(gray)
print("에지 픽셀 비율:", round(edges.mean(), 3), "| 원 후보 개수:", len(circles))
if best:
    print("눈1=({},{}) 눈2=({},{}) → 코 추정=({}, {})".format(
        best[0][0], best[0][1], best[1][0], best[1][1], int(nose[0]), int(nose[1])))

# ---------- 한계 체험: 같은 규칙을 45° 회전 사진에 적용 ----------
rot = transform.rotate(gray, 45, resize=True)
edges_r, circles_r, best_r, nose_r = detect_face_by_rules(rot)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(cat); axes[0].set_title("① 원본"); axes[0].axis("off")
axes[1].imshow(edges, cmap="gray"); axes[1].set_title("② Canny 에지 (기본 요소)"); axes[1].axis("off")
draw_result(axes[2], cat, circles, best, nose, "③ 규칙 적용: 눈(초록) + 코(빨강)")
draw_result(axes[3], rot, circles_r, best_r, nose_r, "④ 45° 회전: 같은 규칙 → 오인식")
plt.tight_layout(); plt.show()
print("→ '수평으로 나란한 두 원'이라는 규칙은 회전·자세 변화를 예상하지 못한다 "
      "(구조적 접근의 한계: 규칙 설계가 어렵고 일반화 불가)")

#### **🟠 통계적 접근법** — 템플릿 매칭 & HOG+SVM
HOG+SVM은 **왼쪽 눈으로만 학습**하고, 처음 보는 **오른쪽 눈**을 찾는지 시험합니다.

In [ ]:
# ============================================================
# [예제 2] 통계적 접근법 — (A) 템플릿 매칭  (B) HOG + SVM(2005)
#   공통 아이디어: "사람이 설계한 특징(픽셀값, HOG)"의 통계적 유사도로 찾는다.
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color, feature, transform

cat = data.chelsea()
gray = color.rgb2gray(cat)                              # (300, 451) 0~1

# ------------------------------------------------------------
# (A) 템플릿 매칭: 왼쪽 눈 조각을 '견본'으로 잘라 이미지 전체를 슬라이딩하며 비교
# ------------------------------------------------------------
LEFT_EYE  = (113, 172)    # (y, x) 왼쪽 눈 중심 — 예제1 Hough 결과
RIGHT_EYE = (130, 310)    # (y, x) 오른쪽 눈 중심
R = 32                    # 잘라낼 반경 → 64×64 조각
template = gray[LEFT_EYE[0]-R:LEFT_EYE[0]+R, LEFT_EYE[1]-R:LEFT_EYE[1]+R]


def match(img, tpl):
    """정규화 상호상관(NCC): -1~1, 1에 가까울수록 '통계적으로' 닮은 위치"""
    res = feature.match_template(img, tpl, pad_input=True)  # 각 위치의 유사도 지도
    y, x = np.unravel_index(np.argmax(res), res.shape)
    return res, (x, y), res.max()

# 같은 사진 / 1.6배 확대 / 90° 회전 사진에서 '같은 템플릿'으로 찾아보기
tests = {
    "원본":      gray,
    "1.6배 확대": transform.rescale(gray, 1.6, anti_aliasing=True),
    "90° 회전":  transform.rotate(gray, 90, resize=True),
}
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(template, cmap="gray")
axes[0].set_title("템플릿(왼쪽 눈)")

res0, _, _ = match(gray, template)
axes[1].imshow(res0, cmap="jet")
axes[1].set_title("유사도 지도(NCC)")
# cmap="jet": 가장 낮은 값은 파란색, 중간값은 초록색/노란색, 가장 높은 값은 빨간색으로 표현되어,
# NCC(정규화 상호상관, Normalized Cross-Correlation) 연산 결과에서
# 템플릿과 가장 유사도가 높은 위치(Peak)가 붉은색으로 뚜렷하게 강조됩니다

for ax, (name, img) in zip(axes[2:], tests.items()):
    _, (x, y), score = match(img, template)
    ax.imshow(img, cmap="gray")
    ax.add_patch(plt.Rectangle((x-R, y-R), 2*R, 2*R, fill=False, ec="red", lw=3))
    ax.set_title(f"{name}  최고 유사도={score:.2f}")
    print(f"[템플릿 매칭] {name:6s} → 최고 유사도 {score:.3f}")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
# → 원본은 1.00, 크기·회전이 바뀌면 점수가 뚝 떨어지고 엉뚱한 곳을 가리킨다
#   (NCC는 평균·분산으로 정규화하므로 '전체 밝기' 변화에는 비교적 강하다 — 직접 gray*0.3으로 확인해 보자)


# ------------------------------------------------------------
# (B) HOG + SVM: 에지 방향 히스토그램(HOG) 특징 + 선형 SVM 분류기
#   ★ 학습은 '왼쪽 눈'으로만! → 처음 보는 '오른쪽 눈'을 찾을 수 있는지 시험
# ------------------------------------------------------------
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
WIN = 64                                                # 슬라이딩 윈도우 크기

def hog_of(patch):
    """64×64 조각 → HOG 벡터(8×8 셀, 9개 방향 구간)"""
    return feature.hog(patch, orientations=9, pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2), block_norm="L2-Hys")

def crop(img, cy, cx, size):
    """(cy, cx) 중심에서 size×size로 잘라 WIN×WIN으로 맞춤(크기 변화 증강용)"""
    h = size // 2
    p = img[max(cy-h, 0):cy+h, max(cx-h, 0):cx+h]
    return transform.resize(p, (WIN, WIN), anti_aliasing=True)

# 양성(눈): 왼쪽 눈 주변을 조금씩 흔들고(위치·크기) 좌우 반전까지 → 200장
pos = []
for _ in range(200):
    dy, dx = rng.integers(-5, 6, size=2)
    size = int(WIN * rng.uniform(0.85, 1.15))
    p = crop(gray, LEFT_EYE[0]+dy, LEFT_EYE[1]+dx, size)
    pos.append(p[:, ::-1] if rng.random() < 0.5 else p)

# 음성(눈 아님): 고양이 사진에서 두 눈과 겹치지 않는 곳 + 다른 사진(커피·로켓·벽돌·풀)
def far_from_eyes(y, x):
    return all(np.hypot(y-ey, x-ex) > 45 for ey, ex in (LEFT_EYE, RIGHT_EYE))
neg = []
while len(neg) < 400:
    y, x = rng.integers(R, gray.shape[0]-R), rng.integers(R, gray.shape[1]-R)
    if far_from_eyes(y, x):
        neg.append(crop(gray, y, x, WIN))
for img in [data.coffee(), data.rocket(), data.brick(), data.grass()]:
    g = color.rgb2gray(img) if img.ndim == 3 else img / 255.0
    for _ in range(100):
        y, x = rng.integers(R, g.shape[0]-R), rng.integers(R, g.shape[1]-R)
        neg.append(crop(g, y, x, WIN))

X = np.array([hog_of(p) for p in pos + neg])            # (N, 1764) 특징 행렬
y = np.array([1]*len(pos) + [0]*len(neg))
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
svm = LinearSVC(C=0.01, max_iter=5000).fit(X_tr, y_tr)
print(f"\n[HOG+SVM] 특징 차원={X.shape[1]}, 학습 {len(X_tr)}장 / 검증 정확도 {svm.score(X_te, y_te):.3f}")

def sliding_window(img, step=6):
    """이미지 전체를 윈도우로 훑으며 SVM 점수(결정 함수 값) 지도를 만든다"""
    ys = range(0, img.shape[0]-WIN, step); xs = range(0, img.shape[1]-WIN, step)
    feats = [hog_of(img[y:y+WIN, x:x+WIN]) for y in ys for x in xs]
    scores = svm.decision_function(np.array(feats)).reshape(len(ys), len(xs))
    return scores, list(ys), list(xs)

def top_k_boxes(scores, ys, xs, k=2, min_gap=60):
    """점수 높은 순으로 서로 떨어진 k개 상자 고르기(간단한 비최대 억제, NMS)"""
    order = np.dstack(np.unravel_index(np.argsort(-scores, axis=None), scores.shape))[0]
    boxes = []
    for i, j in order:
        y0, x0 = ys[i], xs[j]
        if all(np.hypot(y0-b[0], x0-b[1]) > min_gap for b in boxes):
            boxes.append((y0, x0, scores[i, j]))
        if len(boxes) == k: break
    return boxes

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
hog_vec, hog_img = feature.hog(template, orientations=9, pixels_per_cell=(8, 8),
                               cells_per_block=(2, 2), visualize=True)
axes[0].imshow(hog_img, cmap="gray"); axes[0].set_title("HOG 시각화(사람이 설계한 특징)")
for ax, (name, img) in zip(axes[1:], tests.items()):
    scores, ys, xs = sliding_window(img)
    ax.imshow(img, cmap="gray")
    for (y0, x0, s) in top_k_boxes(scores, ys, xs):
        c = "lime" if s > 0 else "red"          # SVM 점수 > 0 이면 '눈'이라고 판정
        ax.add_patch(plt.Rectangle((x0, y0), WIN, WIN, fill=False, ec=c, lw=3))
        ax.text(x0, y0-4, f"{s:.2f}", color=c, fontsize=11)
        print(f"[HOG+SVM] {name:6s} 상자 중심=({x0+WIN//2},{y0+WIN//2}) 점수={s:+.2f}"
              f" → {'눈' if s > 0 else '눈 아님'}")
    ax.set_title(f"{name}: 상위 2개(초록=눈 판정)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
# → 원본에서는 학습에 쓰지 않은 '오른쪽 눈'도 찾는다(템플릿보다 일반화↑).
#   하지만 윈도우 크기(64)·방향이 고정된 특징이라 확대·큰 회전에는 점수가 떨어진다.
#   (실전에서는 이미지 피라미드로 여러 크기를 훑는다 — 도전 과제)


#### **🟠 신경망 접근법** — LeNet 스타일 CNN(End-to-End) & 사전학습 ResNet-18
예제 2와 **같은 학습 규칙(왼쪽 눈만)**. 차이: ① HOG 없이 원시 픽셀 입력 ② 회전·크기 변화를 '데이터 증강'으로 학습

In [ ]:
# ============================================================
# [예제 3] 신경망(End-to-End) 접근법
#   (A) LeNet(1998) 스타일 CNN을 '원시 픽셀'로 직접 학습 → 필터(특징)+분류 규칙을 동시에 학습
#   (B) ImageNet 사전학습 ResNet-18(2015) / AlexNet(2012)으로 '고양이' 분류
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from skimage import data, color, transform

torch.manual_seed(0)
rng = np.random.default_rng(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

cat = data.chelsea()
gray = color.rgb2gray(cat).astype(np.float32)
LEFT_EYE, RIGHT_EYE, WIN, R = (113, 172), (130, 310), 64, 32

# ------------------------------------------------------------
# (A-1) 데이터: 예제2와 같은 규칙 → '왼쪽 눈'만 양성으로 학습, 오른쪽 눈은 시험용
#   차이점 ①: HOG 같은 '사람이 설계한 특징' 없이 픽셀 그대로 입력
#   차이점 ②: 회전·크기 변화를 '규칙'이 아니라 '데이터 증강'으로 가르친다
# ------------------------------------------------------------
def crop_aug(img, cy, cx, size, angle):
    """(cy,cx) 중심 size×size 조각을 angle만큼 돌린 뒤 64×64로 맞춤"""
    h = int(size * 0.75)                       # 회전 시 모서리가 비지 않게 넉넉히 자름
    p = img[max(cy-h, 0):cy+h, max(cx-h, 0):cx+h]
    p = transform.rotate(p, angle, mode="reflect")
    c0, c1 = p.shape[0] // 2, p.shape[1] // 2
    p = p[max(c0-size//2, 0):c0+size//2, max(c1-size//2, 0):c1+size//2]
    return transform.resize(p, (WIN, WIN), anti_aliasing=True).astype(np.float32)

pos = [crop_aug(gray, LEFT_EYE[0]+rng.integers(-5, 6), LEFT_EYE[1]+rng.integers(-5, 6),
                int(WIN*rng.uniform(0.7, 1.7)), rng.uniform(0, 360)) for _ in range(800)]
pos = [p[:, ::-1].copy() if rng.random() < 0.5 else p for p in pos]   # 좌우 반전

def far_from_eyes(y, x):
    return all(np.hypot(y-ey, x-ex) > 45 for ey, ex in (LEFT_EYE, RIGHT_EYE))

neg = []
while len(neg) < 1200:
    y, x = rng.integers(40, gray.shape[0]-40), rng.integers(40, gray.shape[1]-40)
    if far_from_eyes(y, x):
        neg.append(crop_aug(gray, y, x, int(WIN*rng.uniform(0.7, 1.7)), rng.uniform(0, 360)))

for img in [data.coffee(), data.rocket(), data.brick(), data.grass()]:
    g = (color.rgb2gray(img) if img.ndim == 3 else img / 255.0).astype(np.float32)
    for _ in range(200):
        y, x = rng.integers(50, g.shape[0]-50), rng.integers(50, g.shape[1]-50)
        neg.append(crop_aug(g, y, x, int(WIN*rng.uniform(0.7, 1.7)), rng.uniform(0, 360)))

X = torch.tensor(np.array(pos + neg))[:, None]          # (N, 1, 64, 64) = NCHW
y = torch.tensor([1]*len(pos) + [0]*len(neg))
perm = torch.randperm(len(X)); X, y = X[perm], y[perm]
n_tr = int(len(X) * 0.8)
print(f"데이터: 양성 {len(pos)} / 음성 {len(neg)} → 학습 {n_tr}, 검증 {len(X)-n_tr}")

# ------------------------------------------------------------
# (A-2) 모델: LeNet-5(LeCun et al., 1998) 구조를 64×64 입력에 맞게 조금 키운 버전
#   Conv(필터 학습) → Pool → Conv → Pool → FC → 출력(눈/아님)
# ------------------------------------------------------------
class LeNetEye(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 5, padding=2)    # 5×5 필터 8개: 처음엔 무작위 숫자!
        self.conv2 = nn.Conv2d(8, 16, 5, padding=2)
        self.conv3 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn1, self.bn2, self.bn3 = nn.BatchNorm2d(8), nn.BatchNorm2d(16), nn.BatchNorm2d(32)
        self.fc1 = nn.Linear(32 * 8 * 8, 64)
        self.fc2 = nn.Linear(64, 2)
    def forward(self, x):
        # (현대식 보강) BatchNorm(2015): 층마다 값의 분포를 맞춰 학습을 안정화
        x = F.max_pool2d(F.relu(self.bn1(self.conv1(x))), 2)    # 64 → 32
        x = F.max_pool2d(F.relu(self.bn2(self.conv2(x))), 2)    # 32 → 16
        x = F.max_pool2d(F.relu(self.bn3(self.conv3(x))), 2)    # 16 → 8
        x = F.dropout(F.relu(self.fc1(x.flatten(1))), 0.3, self.training)
        return self.fc2(x)

model = LeNetEye().to(device)
print("LeNetEye 파라미터 수:", sum(p.numel() for p in model.parameters()))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(8):                                  # CPU에서도 1분 이내
    model.train()
    idx = torch.randperm(n_tr)
    for b in range(0, n_tr, 64):
        bi = idx[b:b+64]
        loss = F.cross_entropy(model(X[bi].to(device)), y[bi].to(device))
        opt.zero_grad(); loss.backward(); opt.step()    # 오차역전파 → 필터까지 함께 갱신
    model.eval()
    with torch.no_grad():
        acc = (model(X[n_tr:].to(device)).argmax(1).cpu() == y[n_tr:]).float().mean()
    print(f"epoch {epoch+1}: loss={loss.item():.3f}  검증 정확도={acc:.3f}")

# ------------------------------------------------------------
# (A-3) 슬라이딩 윈도우 검출 — 예제2와 똑같은 3가지 시험 사진
# ------------------------------------------------------------
@torch.no_grad()
def cnn_sliding_window(img, step=6):
    ys = list(range(0, img.shape[0]-WIN, step)); xs = list(range(0, img.shape[1]-WIN, step))
    t = torch.tensor(img, dtype=torch.float32)[None, None]          # (1,1,H,W)
    patches = t.unfold(2, WIN, step).unfold(3, WIN, step)            # (1,1,nY,nX,64,64)
    patches = patches[0, 0, :len(ys), :len(xs)].reshape(-1, 1, WIN, WIN)
    prob = torch.cat([F.softmax(model(patches[i:i+512].to(device)), 1)[:, 1].cpu()
                      for i in range(0, len(patches), 512)])
    return prob.reshape(len(ys), len(xs)).numpy(), ys, xs

def top_k_boxes(scores, ys, xs, k=2, min_gap=60):
    order = np.dstack(np.unravel_index(np.argsort(-scores, axis=None), scores.shape))[0]
    boxes = []
    for i, j in order:
        if all(np.hypot(ys[i]-b[0], xs[j]-b[1]) > min_gap for b in boxes):
            boxes.append((ys[i], xs[j], scores[i, j]))
        if len(boxes) == k: break
    return boxes

tests = {"원본": gray,
         "1.6배 확대": transform.rescale(gray, 1.6, anti_aliasing=True).astype(np.float32),
         "90° 회전": transform.rotate(gray, 90, resize=True).astype(np.float32)}
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
w = model.conv1.weight.detach().cpu()[:, 0]                         # (8, 5, 5)
axes[0].imshow(np.hstack([np.pad(f.numpy(), 1, constant_values=np.nan) for f in w]), cmap="bwr")
axes[0].set_title("학습된 conv1 필터 8개\n(사람이 설계 X, 데이터로 학습)")
for ax, (name, img) in zip(axes[1:], tests.items()):
    prob, ys, xs = cnn_sliding_window(img)
    ax.imshow(img, cmap="gray")
    for (y0, x0, p) in top_k_boxes(prob, ys, xs):
        c = "lime" if p > 0.5 else "red"
        ax.add_patch(plt.Rectangle((x0, y0), WIN, WIN, fill=False, ec=c, lw=3))
        ax.text(x0, y0-4, f"{p:.2f}", color=c, fontsize=11)
        print(f"[CNN] {name:6s} 상자 중심=({x0+WIN//2},{y0+WIN//2}) 눈 확률={p:.2f}")
    ax.set_title(f"{name}: 상위 2개(초록=눈 판정)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

# 첫 층 '특징 지도' 보기: 필터가 에지·밝기 변화에 반응 → Canny를 스스로 배운 셈
with torch.no_grad():
    model.eval()   # 실제 순전파와 같게: conv1 → BatchNorm → ReLU
    fmap = F.relu(model.bn1(model.conv1(torch.tensor(gray)[None, None].to(device))))[0].cpu()
fig, axes = plt.subplots(1, 8, figsize=(20, 2.6))
for i, ax in enumerate(axes):
    ax.imshow(fmap[i], cmap="magma"); ax.set_title(f"필터 {i} 반응"); ax.axis("off")
plt.tight_layout(); plt.show()

# ------------------------------------------------------------
# (B) 사전학습 대형 CNN: ImageNet(1,000종) 분류 — '고양이' 판정 (인터넷 필요)
# ------------------------------------------------------------
from torchvision.models import alexnet, resnet18, ResNet18_Weights

count = lambda m: sum(p.numel() for p in m.parameters())
print("\n[모델 규모 비교] LeNetEye: {:,} | AlexNet(2012): {:,} | ResNet-18(2015): {:,}".format(
    count(model), count(alexnet(weights=None)), count(resnet18(weights=None))))

try:
    weights = ResNet18_Weights.DEFAULT                # 첫 실행 시 약 45MB 내려받음
    net = resnet18(weights=weights).eval().to(device)
    preprocess = weights.transforms()                 # Resize→CenterCrop→정규화
    labels = weights.meta["categories"]
    rgb = torch.from_numpy(cat).permute(2, 0, 1)      # HWC → CHW (uint8 그대로 넘기면 됨)
    for name, img in [("원본", rgb), ("90° 회전", torch.rot90(rgb, 1, dims=(1, 2)))]:
        with torch.no_grad():
            p = F.softmax(net(preprocess(img)[None].to(device)), 1)[0].cpu()
        top = torch.topk(p, 3)
        print(f"[ResNet-18] {name}: " + ", ".join(
            f"{labels[i]} {v*100:.1f}%" for v, i in zip(top.values, top.indices)))
except Exception as e:
    print("사전학습 가중치를 내려받지 못했습니다(인터넷 확인):", type(e).__name__)




---



### **[실습] 사람이 설계한 필터 실제 사진에 적용하기** — Sobel(1968)·Canny(1986)
Skimage의 data.chelsea() 데이터셋을 이용하여 Sobel 커널을 **OpenCV**와 **PyTorch `F.conv2d`** 로 각각 적용해, 같은 연산임을 확인합니다.

In [ ]:
from skimage import data

cat = data.chelsea()

gray = cv2.cvtColor(cat, cv2.COLOR_RGB2GRAY).astype(np.float32)

sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=np.float32)   # 세로 경계(가로 방향 밝기 변화)
sobel_y = sobel_x.T                                   # 가로 경계

# (1) OpenCV filter2D 도 교차상관입니다.
gx_cv = cv2.filter2D(gray, -1, sobel_x, borderType=cv2.BORDER_CONSTANT)
# (2) PyTorch 합성곱 (padding=1 → 같은 크기)
gx_pt = F.conv2d(torch.tensor(gray)[None, None], torch.tensor(sobel_x)[None, None], padding=1)[0, 0].numpy()
print("OpenCV vs PyTorch 최대 차이:", np.abs(gx_cv - gx_pt).max())


gy = cv2.filter2D(gray, -1, sobel_y, borderType=cv2.BORDER_CONSTANT)
mag = np.sqrt(gx_cv**2 + gy**2)                       # 경계 강도
canny = cv2.Canny(cv2.GaussianBlur(gray.astype(np.uint8), (5, 5), 0), 100, 200)

fig, axes = plt.subplots(1, 5, figsize=(17, 3.4))
for ax, img, title in zip(axes, [gray, gx_cv, gy, mag, canny],
                          ["흑백", "Sobel-x (세로 경계)", "Sobel-y (가로 경계)", "경계 강도", "Canny (1986)"]):
    ax.imshow(np.abs(img) if title.startswith("Sobel") else img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

### **[실습] 무작위 커널 적용해보기**
Sobel 커널 대신 **3×3 무작위 커널 8개**를 만들어 고양이 사진에 적용해 보고 어떤 커널이 '쓸모 있어' 보이는지 확인하기

In [ ]:
# ============================================================
# 무작위 3×3 커널 8개 vs Sobel — "쓸모 있는 필터"는 무엇이 다른가?
#   CNN의 conv1 필터는 처음에 바로 이런 '무작위 숫자'에서 출발한다.
# ============================================================
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from skimage import data

cat = data.chelsea()
gray = cv2.cvtColor(cat, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0   # 0~1로 정규화

def draw_kernel(ax, k, title, vlim=2.5):
    """3×3 커널을 색(빨강=양수, 파랑=음수)으로 그리고, 각 칸에 커널값 숫자를 표시"""
    k = np.asarray(k)
    ax.imshow(k, cmap="bwr", vmin=-vlim, vmax=vlim)
    for (r, c), v in np.ndenumerate(k):                 # (행, 열) 위치마다 숫자 쓰기
        color = "white" if abs(v) > vlim * 0.6 else "black"   # 진한 칸은 흰 글씨
        ax.text(c, r, f"{v:+.2f}", ha="center", va="center",
                fontsize=10, fontweight="bold", color=color)
    total = round(float(k.sum()), 2) + 0.0              # 부동소수 오차로 생기는 -0.00 표시 방지
    ax.set_title(f"{title}\n합={total:+.2f}", fontsize=10)
    ax.axis("off")

def show(ax, r):
    """응답의 크기 |r|를 상위 1% 값 기준으로 밝기 조정해 표시(약한 에지도 보이게)"""
    a = np.abs(r)
    ax.imshow(a, cmap="gray", vmin=0, vmax=np.percentile(a, 99))

# ------------------------------------------------------------
# [1단계: 최소 예제] 무작위 커널 8개를 한 번에 적용
# ------------------------------------------------------------
torch.manual_seed(42)                                  # 같은 결과를 보려면 고정
K = torch.randn(8, 1, 3, 3)                            # (C_out=8, C_in=1, 3, 3): 표준정규분포 숫자
x = torch.tensor(gray)[None, None]                     # (N=1, C=1, H, W)
out = F.conv2d(x, K, padding=1)[0].numpy()             # (8, H, W): 커널 8개의 응답 지도

fig, axes = plt.subplots(2, 8, figsize=(22, 6))
for i in range(8):
    draw_kernel(axes[0, i], K[i, 0].numpy(), f"커널 {i}")         # 색 + 칸별 숫자 + 합
    show(axes[1, i], out[i])                                        # 반응의 '크기'만 표시
    axes[1, i].set_title(f"응답 {i}")
for ax in axes.ravel(): ax.axis("off")
plt.suptitle("무작위 3×3 커널 8개(위)와 고양이 사진에 적용한 결과(아래)")
plt.tight_layout(); plt.show()


In [ ]:
# ------------------------------------------------------------
# [2단계: 확장] '쓸모'를 숫자로 재 보기
#   기준 정답: Sobel로 구한 경계 강도(mag) — 사람이 설계한 에지 검출기
# ------------------------------------------------------------
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
gx = cv2.filter2D(gray, -1, sobel_x, borderType=cv2.BORDER_CONSTANT)
gy = cv2.filter2D(gray, -1, sobel_x.T, borderType=cv2.BORDER_CONSTANT)
mag = np.sqrt(gx**2 + gy**2)

def corr(a, b):
    """피어슨 상관계수: 두 그림이 '같은 곳에서 밝은가' (−1~1)"""
    a, b = a.ravel() - a.mean(), b.ravel() - b.mean()
    return float((a @ b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

inner = (slice(1, -1), slice(1, -1))                   # 테두리 1픽셀 제외(0-패딩 영향 제거)
rows = []
for i in range(8):
    k = K[i, 0].numpy()
    r = out[i]
    dc_ratio  = abs(k.sum()) / np.abs(k).sum()         # ① 합 성분 비율: 1에 가까우면 '흐린 복사본'
    edge_corr = corr(np.abs(r)[inner], mag[inner])     # ② 경계 강도와 닮은 정도
    gray_corr = corr(np.abs(r)[inner], gray[inner])    # ③ 원래 밝기와 닮은 정도
    x_pref = corr(np.abs(r)[inner], np.abs(gx)[inner]) # ④ 세로 경계 선호
    y_pref = corr(np.abs(r)[inner], np.abs(gy)[inner]) #    가로 경계 선호
    kind = ("에지 검출기 ✅" if edge_corr > 0.5 and dc_ratio < 0.3 else
            "밝기 복사(흐림) ❌" if gray_corr > 0.7 else "섞임 ⚠️")
    rows.append((i, k.sum(), dc_ratio, edge_corr, gray_corr,
                 "세로" if x_pref > y_pref else "가로", kind))

print(f"{'커널':>4} | {'합':>6} | {'합 비율':>6} | {'경계 닮음':>7} | {'밝기 닮음':>7} | 선호 | 판정")
print("-" * 72)
for i, s, dc, ec, gc, d, kind in sorted(rows, key=lambda t: -t[3]):
    print(f"{i:>4} | {s:>6.2f} | {dc:>7.2f} | {ec:>9.2f} | {gc:>9.2f} | {d}  | {kind}")
print(f"{'Sobel':>4} | {sobel_x.sum():>6.2f} | {0:>7.2f} | {1:>9.2f} | {corr(mag[inner], gray[inner]):>9.2f} |  –   | 기준")


In [ ]:
# ------------------------------------------------------------
# [3단계: 실험] 무작위 커널에서 '합(평균)'만 빼면? → 에지 검출기가 된다
# ------------------------------------------------------------
K0 = K - K.mean(dim=(2, 3), keepdim=True)              # 각 커널의 9개 숫자 합을 0으로
out0 = F.conv2d(x, K0, padding=1)[0].numpy()
fig, axes = plt.subplots(4, 8, figsize=(22, 11))
for i in range(8):
    draw_kernel(axes[0, i], K[i, 0].numpy(),  f"원래 커널 {i}")
    show(axes[1, i], out[i])
    axes[1, i].set_title(f"원래 응답: 경계 {corr(np.abs(out[i])[inner], mag[inner]):.2f}", fontsize=10)
    draw_kernel(axes[2, i], K0[i, 0].numpy(), f"평균 뺀 커널 {i}")
    show(axes[3, i], out0[i])
    axes[3, i].set_title(f"합=0 응답: 경계 {corr(np.abs(out0[i])[inner], mag[inner]):.2f}", fontsize=10)
for ax in axes.ravel(): ax.axis("off")
plt.suptitle("1·2행: 무작위 커널과 응답 / 3·4행: 평균을 뺀 커널(합=0)과 응답 — '경계'는 Sobel 경계 강도와의 상관")
plt.tight_layout(); plt.show()

### **[실습] 템플릿 매칭(통계적 접근)의 한계 체험하기**
앞에서 사용한 고양이 이미지에서  고양이 **눈** 영역을 잘라 템플릿으로 쓰고, 원본·축소·회전 이미지에서 찾아봅니다.

In [ ]:
from skimage import data

cat = data.chelsea()

gray = cv2.cvtColor(cat, cv2.COLOR_RGB2GRAY).astype(np.float32)

template = gray[95:135, 160:205].astype(np.uint8)       # 왼쪽 눈 부근(40×45)
h, w = template.shape

def best_match(img_u8, tpl):
    """정규화 상관계수(TM_CCOEFF_NORMED) 최고 점수와 위치."""
    res = cv2.matchTemplate(img_u8, tpl, cv2.TM_CCOEFF_NORMED)
    _, max_val, _, max_loc = cv2.minMaxLoc(res)
    return max_val, max_loc

g8 = gray.astype(np.uint8)
variants = {
    "원본": g8,
    "70% 축소": cv2.resize(g8, None, fx=0.7, fy=0.7),
    "30° 회전": cv2.warpAffine(g8, cv2.getRotationMatrix2D((g8.shape[1]/2, g8.shape[0]/2), 30, 1.0),
                               (g8.shape[1], g8.shape[0])),
    "밝기 낮춤": (g8 * 0.4).astype(np.uint8),
}
fig, axes = plt.subplots(1, len(variants) + 1, figsize=(17, 3.4))
axes[0].imshow(template, cmap="gray"); axes[0].set_title("템플릿(눈)"); axes[0].axis("off")
for ax, (name, img) in zip(axes[1:], variants.items()):
    score, (x0, y0) = best_match(img, template)
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    cv2.rectangle(vis, (x0, y0), (x0 + w, y0 + h), (255, 0, 0), 3)
    ax.imshow(vis); ax.set_title(f"{name}\n점수 {score:.2f}"); ax.axis("off")
    print(f"{name:8s} 최고 유사도 = {score:.3f}  위치 = {(x0, y0)}")
plt.tight_layout(); plt.show()

**관찰** : 밝기 변화에는 강하지만(정규화 덕분), **크기·회전이 바뀌면 점수가 떨어지거나 엉뚱한 곳을 찾습니다.**
→ "필터를 사람이 정하지 말고 **데이터로부터 학습**하자" = CNN의 출발점



---



## **1-2. 이미지는 왜 CNN인가?**
-  CNN, Convolutional Neural Network

### **1. MLP vs 합성곱을 이용한 파라미터 차이**

In [ ]:
import torch.nn as nn

count = lambda m: sum(p.numel() for p in m.parameters())   # 파라미터 개수 세기

for name, (C, H, W) in {"MNIST 28×28×1": (1, 28, 28), "사진 224×224×3": (3, 224, 224)}.items():
    mlp  = nn.Linear(C * H * W, 1000)              # 픽셀을 한 줄로 펴서 1,000개 뉴런에 전부 연결
    conv = nn.Conv2d(C, 64, kernel_size=3)         # 3×3 필터 64개를 모든 위치에 재사용
    print(f"{name}:  MLP {count(mlp):>12,}  |  Conv {count(conv):>6,}")

In [ ]:
import torch, torch.nn as nn

conv = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=3, padding=1)
print("가중치 모양:", tuple(conv.weight.shape))   # (64, 1, 3, 3) = (필터 개수, 입력 채널, 높이, 너비)
print("편향 모양  :", tuple(conv.bias.shape))     # (64,)        = 필터마다 1개
print("파라미터 수:", conv.weight.numel() + conv.bias.numel())   # 640

x = torch.randn(1, 1, 28, 28)                     # MNIST 1장 (N, C, H, W)
print("출력 모양  :", tuple(conv(x).shape))       # (1, 64, 28, 28) → 특징 지도 64장

### **2. CNN의 전체구조 - MNIST 예제**
- [주의!] 런타임유형을 T4로 설정후 실행 (CPU 사용시 오래 걸림)

In [ ]:
# ============================================================
# 그림의 CNN을 손글씨 숫자(MNIST)에 적용하기
#   입력 1×28×28 → [Conv3×3+BN+ReLU] 32×28×28 → [MaxPool2] 32×14×14
#               → [Conv3×3+BN+ReLU] 64×14×14 → [MaxPool2] 64×7×7
#               → [Flatten] 3136 → [Linear] 10
#   코랩: 런타임 > 런타임 유형 변경 > T4 GPU 권장 (CPU도 3에폭 약 2~4분)
# ============================================================
import time
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("사용 장치:", device)

# ------------------------------------------------------------
# 1. 데이터 준비: 학습 60,000장 / 시험 10,000장
#   ToTensor   : PIL(0~255, H×W) → 텐서(0~1, 1×28×28)
#   Normalize  : MNIST 전체 평균 0.1307, 표준편차 0.3081로 표준화 (평균 0, 분산 1 근처)
# ------------------------------------------------------------
tf = transforms.Compose([transforms.ToTensor(),
                         transforms.Normalize((0.1307,), (0.3081,))])
train_ds = datasets.MNIST("data", train=True,  download=True, transform=tf)
test_ds  = datasets.MNIST("data", train=False, download=True, transform=tf)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=1000)
print(f"학습 {len(train_ds):,}장 / 시험 {len(test_ds):,}장, 한 장 모양 {tuple(train_ds[0][0].shape)}")

# ------------------------------------------------------------
# 2. 모델: 그림의 박스 하나 = 코드 한 줄
# ------------------------------------------------------------
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(                          # 특징 추출기: 에지 → 질감 → 부분
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),   # 1×28×28  → 32×28×28
            nn.MaxPool2d(2),                                                 # 32×28×28 → 32×14×14
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),  # 32×14×14 → 64×14×14
            nn.MaxPool2d(2),                                                 # 64×14×14 → 64×7×7
        )
        self.classifier = nn.Sequential(                        # 분류기
            nn.Flatten(),                                       # 64×7×7 → 3136
            nn.Linear(64 * 7 * 7, 10),                          # 3136 → 10 (숫자 0~9 점수)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = MnistCNN().to(device)

# 그림과 같은지 확인: 가짜 입력 1장을 흘려보내며 층마다 모양과 파라미터 수 출력
print(f"\n{'층':<14}{'출력 모양':<18}{'파라미터':>10}")
print("-" * 42)
h = torch.zeros(1, 1, 28, 28, device=device)
print(f"{'입력':<14}{str(tuple(h.shape[1:])):<18}{'':>10}")
for layer in list(model.features) + list(model.classifier):
    h = layer(h)
    n = sum(p.numel() for p in layer.parameters())
    print(f"{layer.__class__.__name__:<14}{str(tuple(h.shape[1:])):<18}{n:>10,}")
print(f"{'합계':<32}{sum(p.numel() for p in model.parameters()):>10,}")

# ------------------------------------------------------------
# 3. 학습 / 평가 함수
# ------------------------------------------------------------
def evaluate(model, dl):
    model.eval()                                    # BN: 학습 중 모은 평균·분산 사용
    correct = 0
    with torch.no_grad():
        for xb, yb in dl:
            correct += (model(xb.to(device)).argmax(1).cpu() == yb).sum().item()
    return correct / len(dl.dataset)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
for epoch in range(1, 4):                           # 3에폭이면 약 99% 근처
    model.train()                                   # BN: 미니배치 평균·분산 사용
    t0, total_loss = time.time(), 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        loss = F.cross_entropy(model(xb), yb)       # 소프트맥스 + 교차 엔트로피
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item() * len(xb)
    acc = evaluate(model, test_dl)
    history.append((total_loss / len(train_ds), acc))
    print(f"epoch {epoch}: 평균 손실 {history[-1][0]:.4f} | 시험 정확도 {acc*100:.2f}% | {time.time()-t0:.0f}초")

# ------------------------------------------------------------
# 4. 결과 보기 ① 틀린 숫자들
# ------------------------------------------------------------
model.eval()
xs, ys = next(iter(test_dl))                        # 시험 1,000장
with torch.no_grad():
    probs = F.softmax(model(xs.to(device)), 1).cpu()
pred = probs.argmax(1)
wrong = (pred != ys).nonzero().flatten()[:10]
print(f"\n시험 {len(ys):,}장 중 틀린 개수: {(pred != ys).sum().item()}")
fig, axes = plt.subplots(1, 10, figsize=(15, 2.2))
for ax, i in zip(axes, wrong):
    ax.imshow(xs[i, 0], cmap="gray")
    ax.set_title(f"정답 {ys[i]} → 예측 {pred[i]}\n({probs[i, pred[i]]*100:.0f}%)", fontsize=9)
for ax in axes: ax.axis("off")
plt.suptitle("CNN이 틀린 숫자 (사람도 헷갈리는 경우가 많다)"); plt.tight_layout(); plt.show()

# ------------------------------------------------------------
# 5. 결과 보기 ② 학습된 첫 층 필터 32개 (3×3)
# ------------------------------------------------------------
w = model.features[0].weight.detach().cpu()[:, 0]          # (32, 3, 3)
fig, axes = plt.subplots(2, 16, figsize=(16, 2.4))
for ax, k in zip(axes.ravel(), w):
    lim = k.abs().max()
    ax.imshow(k, cmap="bwr", vmin=-lim, vmax=lim); ax.axis("off")
plt.suptitle("학습된 conv1 필터 32개 (빨강=양수, 파랑=음수)"); plt.tight_layout(); plt.show()

# ------------------------------------------------------------
# 6. 결과 보기 ③ 숫자 1장이 층을 지나며 바뀌는 모습 (특징 지도)
# ------------------------------------------------------------
x1 = xs[:1].to(device)
maps, h = {}, x1
with torch.no_grad():
    for idx, layer in enumerate(model.features):
        h = layer(h)
        if isinstance(layer, nn.ReLU):                      # 각 Conv 블록의 출력만 저장
            maps[f"블록{len(maps)+1} {tuple(h.shape[1:])}"] = h[0].cpu()
fig, axes = plt.subplots(len(maps), 9, figsize=(14, 3.4))
for r, (name, m) in enumerate(maps.items()):
    axes[r, 0].imshow(xs[0, 0], cmap="gray"); axes[r, 0].set_title("입력", fontsize=9)
    for c in range(1, 9):                                   # 채널 8개만 표시
        axes[r, c].imshow(m[c - 1], cmap="magma")
        axes[r, c].set_title(f"{name}\n채널 {c-1}" if c == 1 else f"채널 {c-1}", fontsize=8)
for ax in axes.ravel(): ax.axis("off")
plt.suptitle(f"숫자 '{ys[0].item()}'의 특징 지도: 블록1(28×28, 획·에지) → 블록2(14×14, 부분 모양)")
plt.tight_layout(); plt.show()

### **2. 합성곱 연산(Convolution)** 손계산→ PyTorch로 검증

커널이 입력 위를 이동하며 **요소별 곱 → 모두 더하기**를 반복합니다.

$$\text{out}[i, j] = \sum_{u=0}^{K-1}\sum_{v=0}^{K-1} X[i+u,\ j+v]\cdot W[u, v] + b$$

In [ ]:
X = np.array([[1, 2, 0, 1, 3],
              [0, 1, 3, 2, 1],
              [2, 1, 0, 1, 0],
              [1, 0, 2, 3, 1],
              [0, 2, 1, 0, 2]], dtype=np.float32)
K = np.array([[1, 0, -1],
              [1, 0, -1],
              [1, 0, -1]], dtype=np.float32)      # 세로 에지 검출 커널

def conv2d_numpy(X, K, stride=1, padding=0):
    """반복문으로 구현한 2차원 합성곱(정확히는 교차상관)."""
    if padding > 0:
        X = np.pad(X, padding)                   # 가장자리에 0 채우기
    k = K.shape[0]
    out_h = (X.shape[0] - k) // stride + 1
    out_w = (X.shape[1] - k) // stride + 1
    out = np.zeros((out_h, out_w), dtype=np.float32)
    for i in range(out_h):
        for j in range(out_w):
            patch = X[i*stride:i*stride+k, j*stride:j*stride+k]   # 커널이 덮는 영역
            out[i, j] = (patch * K).sum()                         # 요소별 곱 → 합
    return out

out_np = conv2d_numpy(X, K)
print("NumPy 손계산 결과:\n", out_np)

# 가운데 위치(1,1)를 한 줄씩 풀어서 계산 → 슬라이드의 강조 위치와 같은 값
print("가운데 값 =", (X[1:4, 1:4] * K).sum())

# PyTorch: 입력(N,C,H,W), 커널(C_out,C_in,K,K) 모양으로 맞춰 줍니다.
out_pt = F.conv2d(torch.tensor(X)[None, None], torch.tensor(K)[None, None])
print("PyTorch 결과:\n", out_pt[0, 0].numpy())
print("두 결과 일치:", np.allclose(out_np, out_pt[0, 0].numpy()))

### **3. 출력 크기 공식** — 스트라이드(S)·패딩(P)

$$O = \left\lfloor \frac{I + 2P - K}{S} \right\rfloor + 1$$

| 기호 | 의미 |
|---|---|
| I | 입력 크기 |
| K | 커널 크기 |
| P | 패딩(가장자리 0 채우기) |
| S | 스트라이드(이동 간격) |

In [ ]:
def out_size(I, K, P=0, S=1):
    """합성곱 출력 크기 공식."""
    return (I + 2 * P - K) // S + 1

cases = [(28, 3, 0, 1),
        (28, 3, 1, 1),
        (28, 3, 1, 2),
        (224, 7, 3, 2),
        (5, 3, 1, 2)]
print(f"{'I':>4} {'K':>3} {'P':>3} {'S':>3} | {'공식':>4} {'nn.Conv2d':>9}")
for I, Kk, P, S in cases:
    conv = nn.Conv2d(1, 1, kernel_size=Kk, padding=P, stride=S)
    real = conv(torch.zeros(1, 1, I, I)).shape[-1]          # 실제 출력 모양
    print(f"{I:>4} {Kk:>3} {P:>3} {S:>3} | {out_size(I, Kk, P, S):>4} {real:>9}")

# 패딩·스트라이드를 바꿔 손계산 함수로도 확인
print("\nP=1, S=1 → 모양", conv2d_numpy(X, K, padding=1).shape, " | P=0, S=2 → 모양", conv2d_numpy(X, K, stride=2).shape)

### **[실습] 출력 크기 연습 문제**

In [ ]:
cases = [(32, 3, 0, 1),
        (32, 3, 1, 1),
        (32, 3, 1, 2),
        (227, 11, 0, 4)]
print(f"{'I':>4} {'K':>3} {'P':>3} {'S':>3} | {'공식':>4} {'nn.Conv2d':>9}")
for I, Kk, P, S in cases:
    conv = nn.Conv2d(1, 1, kernel_size=Kk, padding=P, stride=S)
    real = conv(torch.zeros(1, 1, I, I)).shape[-1]          # 실제 출력 모양
    print(f"{I:>4} {Kk:>3} {P:>3} {S:>3} | {out_size(I, Kk, P, S):>4} {real:>9}")

In [ ]:
import torch
import torch.nn as nn

I, K, P, S = 224, 3, 1, 2

x = torch.zeros(1, 1, I, I)

for i in range(5):
    conv = nn.Conv2d(1, 1, kernel_size=K,
                     padding=P, stride=S)
    x = conv(x)
    print(f"{i+1}회 출력: {x.shape[-1]}×{x.shape[-1]}")

### **4. 다채널 합성곱과 파라미터 수**

커널 하나는 **(입력 채널 × K × K)** 두께의 블록이고, 출력 채널 수만큼 존재합니다.

- $$\text{파라미터 수} = C_{out}\times(C_{in}\times K\times K) + C_{out}$$
- $$(\text{가중치 개수}) + (\text{편향 개수}) = (C_{out} \times C_{in} \times K_h \times K_w) + C_{out}$$ <br>

- 예: 합성곱(3→64, 3×3) 계층의 파라미터 수가 1,792개가 되는 계산 과정
    - 가중치(Weight): $64 \times 3 \times 3 \times 3 = 1,728$개
    - 편향(Bias): $64$개
    - 총합: $1,728 + 64 = 1,792$개

In [ ]:
for c_in, c_out, k in [(1, 32, 3), (3, 64, 3), (32, 64, 3), (3, 64, 7)]:
    conv = nn.Conv2d(c_in, c_out, k)
    n_param = sum(p.numel() for p in conv.parameters())
    print(f"Conv2d({c_in:>2}, {c_out}, {k}) weight 모양={tuple(conv.weight.shape)}  "
          f"파라미터={n_param:>6,}  (공식 {c_out*(c_in*k*k)+c_out:,})")

# 비교: 같은 컬러 사진(224×224×3)을 MLP 은닉층 1,000개에 연결하면?
print(f"\nMLP 첫 층 파라미터 = {224*224*3*1000 + 1000:,} 개  ← 합성곱(3→64, 3×3)은 1,792 개")


### **5. 활성화(ReLU)와 풀링(Pooling)**

In [ ]:
P_in = torch.tensor([[1, 3, 2, 1],
                     [4, 6, 5, 0],
                     [3, 1, 1, 2],
                     [0, 2, 4, 3]], dtype=torch.float32)[None, None]
print("최대 풀링(2×2, S=2):\n", F.max_pool2d(P_in, 2)[0, 0])
print("평균 풀링(2×2, S=2):\n", F.avg_pool2d(P_in, 2)[0, 0])
print("전역 평균 풀링(GAP):", F.adaptive_avg_pool2d(P_in, 1).item())
print("ReLU([-2, -0.5, 0, 1.5]) =", F.relu(torch.tensor([-2, -0.5, 0, 1.5])))



---

